# **Part 0 : Deliverable Guidelines**

# **Part 1 : Data Preperation**

## **1.1 - Import Data Set**

We first found and imported our auditor_sentiment data set from hungging face by connecting to our account with a token.

In [1]:
from datasets import load_dataset
ds = load_dataset("Tydyannes69/auditor_sentiment")

C:\Users\jerem\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Then we wanted to have a look at the data set to see if the importation functioned. So we decided to create a CSV file to be able to visulize it. The data base was already separated in two different data set. A "Train" and "Test" data set. We decided to keep it this way, so we could train our model on the "Train" data set and then measure the quality of our trained model on the "Test" data set.

In [2]:
import pandas as pd

df_train = ds["train"].to_pandas()
df_test = ds["test"].to_pandas()

print(df_train.head())
print(df_test.columns)

                                            sentence  label
0  Altia 's operating profit jumped to EUR 47 mil...      2
1  The agreement was signed with Biohit Healthcar...      2
2  Kesko pursues a strategy of healthy , focused ...      2
3  Vaisala , headquartered in Helsinki in Finland...      1
4  Also , a six-year historic analysis is provide...      1
Index(['sentence', 'label'], dtype='object')


Before doing any descriptive statistics we wanted to combine the "train" and "test" dataset to be able to study our entire dataset

In [3]:
df_combined = pd.concat([df_train, df_test], ignore_index=True)

ydata-cleaning

In [4]:
from ydata_profiling import ProfileReport
profile = ProfileReport(df_combined, title="Profiling Dataset", explorative=True)
profile.to_file("report.html")

C:\Users\jerem\AppData\Local\Temp\ipykernel_7296\252681404.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 665.66it/s]


Doublons

In [5]:
df_combined = df_combined.drop_duplicates()

Sepération

In [6]:
from sklearn.model_selection import train_test_split

df_train_original, df_test_original = train_test_split(df_combined, test_size=0.2,stratify=df_combined["label"],random_state=42)
df_train_original.to_csv("train.csv", index=False)
df_test_original.to_csv("test.csv", index=False)

## **1.2 - Descriptive Statistics**

In [7]:
# Nombre de lignes et colonnes
num_rows = df_combined.shape[0]
num_cols = df_combined.shape[1]

print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_cols}")
print(f"Names of the Columns: {ds['train'].column_names}")

Number of rows: 4840
Number of columns: 2
Names of the Columns: ['sentence', 'label']


### Transformation of the variable "label"

In [8]:
df_combined["label_num"] = df_combined["label"].map({
    0: "negative",
    1: "neutral",
    2: "positive"
})

C:\Users\jerem\AppData\Local\Temp\ipykernel_7296\1716255520.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_combined["label_num"] = df_combined["label"].map({


### Description of the variable "label"

In [9]:
print("Distribution des classes :")
print(df_combined["label_num"].value_counts())

print("\nProportions (%) :")
print((df_combined["label_num"].value_counts(normalize=True) * 100).round(2))

Distribution des classes :
label_num
neutral     2873
positive    1363
negative     604
Name: count, dtype: int64

Proportions (%) :
label_num
neutral     59.36
positive    28.16
negative    12.48
Name: proportion, dtype: float64


In [10]:
import matplotlib.pyplot as plt

df_combined["label_num"].value_counts().plot(kind="bar")
plt.title("Distribution des sentiments")
plt.xlabel("Sentiment")
plt.ylabel("Nombre")
plt.xticks(rotation=0)
plt.show()

C:\Users\jerem\AppData\Local\Temp\ipykernel_7296\3857146993.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## **1.3 - Data Cleaning**

### Text Cleaning

We used the libraries NLTK and SpaCy for text processing and analysis. SpaCy (en_core_web_sm) facilitates advanced tasks such as lemmatization and syntactic analysis. Common words (stopwords) are removed in order to focus on and analyze the most meaningful terms.

Before converting raw text into usable data with CountVectorizer and TfidfVectorizer, it is necessary to clean the data. We therefore decided to standardize the text by converting it to lowercase, then removing special characters, numbers, and extra spaces.

In [11]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

# Stopwords anglais
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # suppression des stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    
    return " ".join(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jerem\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Lemmatize Text

Lemmatization reduces each word to its base form (e.g., “studying” becomes “study”). This reduces the complexity of the vocabulary while preserving the meaning of sentences.

We chose to lemmatize the textual variable "sentence", as it contains longer and meaningful sentences that are important for predicting our target variable.

We chose lemmatization rather than stemming because lemmatization preserves the meaning of words.

Finally, we applied lemmatization before vectorization to prevent the model from considering words like “study”, “studies”, and “studying” as three different terms. By doing so, we reduce computation time and improve the relevance of the results.

In [12]:
import spacy

# Charger modèle anglais
nlp = spacy.load("en_core_web_sm")

def lemmatize_text(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_punct]
    return " ".join(tokens)

Data cleaning and lemmatizing text on train et test dataset

In [13]:
df_train = df_train_original.copy()
df_test = df_test_original.copy()

df_train["sentence_clean"] = df_train_original["sentence"].apply(clean_text).apply(lemmatize_text)
df_test["sentence_clean"] = df_test_original["sentence"].apply(clean_text).apply(lemmatize_text)

In [14]:
df_train.head()

,sentence,label,sentence_clean
883,Currently Alfred A Palmberg is putting the fin...,1,currently alfred palmberg put finishing touch ...
3204,Finnish Rautaruukki has been awarded a contrac...,2,finnish rautaruukki award contract supply inst...
2422,Finnish Bore that is owned by the Rettig famil...,2,finnish bore own rettig family grown recently ...
4831,YIT 's Baltic sales in the first three quarter...,0,yit baltic sale first three quarter total mill...
3371,HKScan is one of the leading food companies in...,1,hkscan one lead food company northern europe h...


### Vectorizing Text

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(df_train["sentence_clean"])
X_test_tfidf = tfidf_vectorizer.transform(df_test["sentence_clean"])

print(tfidf_vectorizer.get_feature_names_out()[:20])
print(X_train_tfidf)
print(X_test_tfidf)

['aaland' 'aalborg' 'aalto' 'aaron' 'aava' 'aazhang' 'ab' 'abb' 'abbott'
 'abc' 'aberration' 'abidjan' 'ability' 'able' 'abloy' 'abn'
 'abovementione' 'abp' 'abroad' 'absentee']
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 43288 stored elements and shape (3872, 6982)>
  Coords	Values
  (0, 1347)	0.20197981198645956
  (0, 179)	0.2969718488733562
  (0, 4357)	0.2829192386627825
  (0, 4861)	0.23209892674688704
  (0, 2113)	0.2829192386627825
  (0, 6330)	0.258896142834846
  (0, 4206)	0.1713594364046685
  (0, 2115)	0.12881017807780107
  (0, 610)	0.2171688519163627
  (0, 1934)	0.2729487530454197
  (0, 5102)	0.2829192386627825
  (0, 532)	0.2829192386627825
  (0, 5024)	0.2829192386627825
  (0, 4793)	0.16504053566727891
  (0, 204)	0.14082542826113775
  (0, 1584)	0.23487304700690959
  (0, 6723)	0.2829192386627825
  (1, 2121)	0.14712493765709633
  (1, 4945)	0.27749717055343054
  (1, 462)	0.2639765971743735
  (1, 1221)	0.19601037452674341
  (1, 5997)	0.23968407624263444
  (1, 2849)	

Top 5 words that come up the most

In [16]:
import numpy as np

# moyenne des scores TF-IDF pour chaque mot (colonne)
mean_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).ravel()

# récupérer les mots
feature_names = tfidf_vectorizer.get_feature_names_out()

# trier du plus important au moins important
top_indices = mean_tfidf.argsort()[::-1]

# afficher les 5 mots les plus importants
for i in top_indices[:5]:
    print(feature_names[i], mean_tfidf[i])

eur 0.04682766637510156
mn 0.03023924413104272
company 0.02602834805254886
profit 0.018822148842717712
sale 0.01881347408395019


### Dimension Reduction

In [17]:
from sklearn.decomposition import TruncatedSVD

# Réduction de dimension
svd = TruncatedSVD(n_components=170, random_state=42)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)

print(X_train_svd.shape)
print(X_test_svd.shape)

(3872, 170)
(968, 170)


# **Part 2 : Classifier**

## **2.1 - Logistic Regression**

Training Model

In [18]:
y_train = df_train["label"]
y_test = df_test["label"]

from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_svd, y_train)

y_pred = clf.predict(X_test_svd)

Testing Model

In [19]:
import time
import psutil
import os
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error
)

y_train = df_train["label"]
y_test = df_test["label"]

process = psutil.Process(os.getpid())
cpu_before_fit = process.cpu_times()

clf = LogisticRegression(max_iter=1000, random_state=42)
start_fit = time.perf_counter()
clf.fit(X_train_svd, y_train)
end_fit = time.perf_counter()

cpu_after_fit = process.cpu_times()

fit_time = end_fit - start_fit
fit_cpu_user = cpu_after_fit.user - cpu_before_fit.user
fit_cpu_sys  = cpu_after_fit.system - cpu_before_fit.system

cpu_before_pred = process.cpu_times()

start_pred = time.perf_counter()
y_pred = clf.predict(X_test_svd)
end_pred = time.perf_counter()

cpu_after_pred = process.cpu_times()

pred_time = end_pred - start_pred
pred_cpu_user = cpu_after_pred.user - cpu_before_pred.user
pred_cpu_sys  = cpu_after_pred.system - cpu_before_pred.system

print("Classification : ")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred))

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)

print("\nErrors : ")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

print("\nPerformance :")
print(f"Entraînement : {fit_time:.4f} s "
      f"(CPU user {fit_cpu_user:.4f}s | sys {fit_cpu_sys:.4f}s)")
print(f"Inférence    : {pred_time:.4f} s "
      f"(CPU user {pred_cpu_user:.4f}s | sys {pred_cpu_sys:.4f}s)")
print(f"Total : {fit_time + pred_time:.4f} s ")

Classification : 
Accuracy : 0.6973
              precision    recall  f1-score   support

           0       0.77      0.31      0.44       121
           1       0.70      0.95      0.80       575
           2       0.67      0.34      0.45       272

    accuracy                           0.70       968
   macro avg       0.71      0.53      0.56       968
weighted avg       0.70      0.70      0.66       968

Confusion matrix :
[[ 37  65  19]
 [  3 545  27]
 [  8 171  93]]

Errors : 
MSE  : 0.3864
RMSE : 0.6216
MAE  : 0.3306

Performance :
Entraînement : 0.0335 s (CPU user 0.0312s | sys 0.0000s)
Inférence    : 0.0003 s (CPU user 0.0000s | sys 0.0000s)
Total : 0.0338 s 


## **2.2 - Gaussian Classifier**

In [20]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error
)

y_train = df_train["label"]
y_test = df_test["label"]

process = psutil.Process(os.getpid())
cpu_before_fit = process.cpu_times()

gnb = GaussianNB()
start_fit = time.perf_counter()
gnb.fit(X_train_svd, y_train)
end_fit = time.perf_counter()

cpu_after_fit = process.cpu_times()

fit_time = end_fit - start_fit
fit_cpu_user = cpu_after_fit.user - cpu_before_fit.user
fit_cpu_sys  = cpu_after_fit.system - cpu_before_fit.system

cpu_before_pred = process.cpu_times()

start_pred = time.perf_counter()
y_pred_gnb = gnb.predict(X_test_svd)
end_pred = time.perf_counter()

cpu_after_pred = process.cpu_times()

pred_time = end_pred - start_pred
pred_cpu_user = cpu_after_pred.user - cpu_before_pred.user
pred_cpu_sys  = cpu_after_pred.system - cpu_before_pred.system

print("Classification : ")
print(f"Accuracy : {accuracy_score(y_test, y_pred_gnb):.4f}")
print(classification_report(y_test, y_pred_gnb))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred_gnb))

mse  = mean_squared_error(y_test, y_pred_gnb)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred_gnb)

print("\nErrors : ")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

print("\nPerformance :")
print(f"Entraînement : {fit_time:.4f} s "
      f"(CPU user {fit_cpu_user:.4f}s | sys {fit_cpu_sys:.4f}s)")
print(f"Inférence    : {pred_time:.4f} s "
      f"(CPU user {pred_cpu_user:.4f}s | sys {pred_cpu_sys:.4f}s)")
print(f"Total : {fit_time + pred_time:.4f} s ")

Classification : 
Accuracy : 0.6188
              precision    recall  f1-score   support

           0       0.35      0.37      0.36       121
           1       0.67      0.83      0.74       575
           2       0.58      0.28      0.38       272

    accuracy                           0.62       968
   macro avg       0.54      0.49      0.49       968
weighted avg       0.61      0.62      0.59       968

Confusion matrix :
[[ 45  65  11]
 [ 53 478  44]
 [ 30 166  76]]

Errors : 
MSE  : 0.5083
RMSE : 0.7129
MAE  : 0.4236

Performance :
Entraînement : 0.0037 s (CPU user 0.0000s | sys 0.0000s)
Inférence    : 0.0010 s (CPU user 0.0000s | sys 0.0000s)
Total : 0.0047 s 


## **2.3 - Bernoulli Classifier**

In [21]:
from sklearn.naive_bayes import BernoulliNB

# Transformer TF-IDF en binaire
X_train_bin = (X_train_tfidf > 0).astype(int)
X_test_bin = (X_test_tfidf > 0).astype(int)

process = psutil.Process(os.getpid())
cpu_before_fit = process.cpu_times()

bnb = BernoulliNB()
start_fit = time.perf_counter()
bnb.fit(X_train_bin, y_train)
end_fit = time.perf_counter()

cpu_after_fit = process.cpu_times()

fit_time = end_fit - start_fit
fit_cpu_user = cpu_after_fit.user - cpu_before_fit.user
fit_cpu_sys  = cpu_after_fit.system - cpu_before_fit.system

cpu_before_pred = process.cpu_times()

start_pred = time.perf_counter()
y_pred_bnb = bnb.predict(X_test_bin)
end_pred = time.perf_counter()

cpu_after_pred = process.cpu_times()

pred_time = end_pred - start_pred
pred_cpu_user = cpu_after_pred.user - cpu_before_pred.user
pred_cpu_sys  = cpu_after_pred.system - cpu_before_pred.system

print("Classification : ")
print(f"Accuracy : {accuracy_score(y_test, y_pred_bnb):.4f}")
print(classification_report(y_test, y_pred_bnb))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred_bnb))

mse  = mean_squared_error(y_test, y_pred_bnb)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred_bnb)

print("\nErrors : ")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

print("\nPerformance :")
print(f"Entraînement : {fit_time:.4f} s "
      f"(CPU user {fit_cpu_user:.4f}s | sys {fit_cpu_sys:.4f}s)")
print(f"Inférence    : {pred_time:.4f} s "
      f"(CPU user {pred_cpu_user:.4f}s | sys {pred_cpu_sys:.4f}s)")
print(f"Total : {fit_time + pred_time:.4f} s ")

Classification : 
Accuracy : 0.6880
              precision    recall  f1-score   support

           0       0.89      0.07      0.12       121
           1       0.71      0.94      0.81       575
           2       0.58      0.43      0.50       272

    accuracy                           0.69       968
   macro avg       0.73      0.48      0.48       968
weighted avg       0.70      0.69      0.64       968

Confusion matrix :
[[  8  64  49]
 [  0 541  34]
 [  1 154 117]]

Errors : 
MSE  : 0.4669
RMSE : 0.6833
MAE  : 0.3636

Performance :
Entraînement : 0.0019 s (CPU user 0.0000s | sys 0.0000s)
Inférence    : 0.0005 s (CPU user 0.0000s | sys 0.0000s)
Total : 0.0024 s 


## **2.4 - Decision Tree**

In [22]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

best_score = 0
best_params = None
best_model = None

# Ressources globales sur toute la recherche
process = psutil.Process(os.getpid())
cpu_before_search = process.cpu_times()
start_search = time.perf_counter()

n_combinations = 0

for max_depth in [5, 10, 15, 20, None]:
    for min_samples_split in [2, 5, 10, 20]:
        for min_samples_leaf in [1, 2, 4, 8]:
            for criterion in ["gini", "entropy"]:
                n_combinations += 1

                dt = DecisionTreeClassifier(
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    min_samples_leaf=min_samples_leaf,
                    criterion=criterion,
                    random_state=42
                )

                dt.fit(X_train_svd, y_train)
                y_pred = dt.predict(X_test_svd)
                acc = accuracy_score(y_test, y_pred)

                if acc > best_score:
                    best_score = acc
                    best_params = {
                        "max_depth": max_depth,
                        "min_samples_split": min_samples_split,
                        "min_samples_leaf": min_samples_leaf,
                        "criterion": criterion
                    }
                    best_model = dt

end_search = time.perf_counter()
cpu_after_search = process.cpu_times()

search_time = end_search - start_search
search_cpu_user = cpu_after_search.user - cpu_before_search.user
search_cpu_sys  = cpu_after_search.system - cpu_before_search.system

print(f"Recherche : {n_combinations} combinaisons testées")
print(f"Meilleure accuracy : {best_score:.4f}")
print(f"Meilleurs params   : {best_params}")
print(f"Temps total recherche : {search_time:.2f} s "
      f"(CPU user {search_cpu_user:.2f}s | sys {search_cpu_sys:.2f}s)")
print(f"Temps moyen / combinaison : {search_time / n_combinations:.3f} s\n")

# --- Évaluation détaillée du meilleur modèle ---
cpu_before_fit = process.cpu_times()
start_fit = time.perf_counter()
best_model.fit(X_train_svd, y_train)
end_fit = time.perf_counter()
cpu_after_fit = process.cpu_times()

fit_time = end_fit - start_fit
fit_cpu_user = cpu_after_fit.user - cpu_before_fit.user
fit_cpu_sys  = cpu_after_fit.system - cpu_before_fit.system

cpu_before_pred = process.cpu_times()
start_pred = time.perf_counter()
y_pred_dt = best_model.predict(X_test_svd)
end_pred = time.perf_counter()
cpu_after_pred = process.cpu_times()

pred_time = end_pred - start_pred
pred_cpu_user = cpu_after_pred.user - cpu_before_pred.user
pred_cpu_sys  = cpu_after_pred.system - cpu_before_pred.system

print("Classification (best model) : ")
print(f"Accuracy : {accuracy_score(y_test, y_pred_dt):.4f}")
print(classification_report(y_test, y_pred_dt))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred_dt))

mse  = mean_squared_error(y_test, y_pred_dt)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred_dt)

print("\nErrors : ")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

print("\nPerformance (best model) :")
print(f"Entraînement : {fit_time:.4f} s "
      f"(CPU user {fit_cpu_user:.4f}s | sys {fit_cpu_sys:.4f}s)")
print(f"Inférence    : {pred_time:.4f} s "
      f"(CPU user {pred_cpu_user:.4f}s | sys {pred_cpu_sys:.4f}s)")
print(f"Total : {fit_time + pred_time:.4f} s ")
print(f"\nArbre : profondeur={best_model.get_depth()}, "
      f"nb feuilles={best_model.get_n_leaves()}")

Recherche : 160 combinaisons testées
Meilleure accuracy : 0.6353
Meilleurs params   : {'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 8, 'criterion': 'entropy'}
Temps total recherche : 85.43 s (CPU user 83.95s | sys 0.83s)
Temps moyen / combinaison : 0.534 s

Classification (best model) : 
Accuracy : 0.6353
              precision    recall  f1-score   support

           0       0.35      0.32      0.34       121
           1       0.71      0.84      0.77       575
           2       0.53      0.34      0.41       272

    accuracy                           0.64       968
   macro avg       0.53      0.50      0.51       968
weighted avg       0.61      0.64      0.61       968

Confusion matrix :
[[ 39  54  28]
 [ 36 484  55]
 [ 35 145  92]]

Errors : 
MSE  : 0.5599
RMSE : 0.7483
MAE  : 0.4298

Performance (best model) :
Entraînement : 0.6126 s (CPU user 0.5781s | sys 0.0000s)
Inférence    : 0.0004 s (CPU user 0.0000s | sys 0.0000s)
Total : 0.6130 s 

Arbre : profondeu

## **2.5 - Random Forest**

In [23]:
best_score = 0
best_params = None
best_model = None

# Ressources globales sur toute la recherche
process = psutil.Process(os.getpid())
cpu_before_search = process.cpu_times()
start_search = time.perf_counter()

n_combinations = 0

for max_depth in [5, 10, 15, 20, None]:
    for min_samples_split in [2, 5, 10, 20]:
        for min_samples_leaf in [1, 2, 4, 8]:
            for criterion in ["gini", "entropy"]:
                n_combinations += 1

                dt = DecisionTreeClassifier(
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    min_samples_leaf=min_samples_leaf,
                    criterion=criterion,
                    random_state=42
                )

                dt.fit(X_train_svd, y_train)
                y_pred = dt.predict(X_test_svd)
                acc = accuracy_score(y_test, y_pred)

                if acc > best_score:
                    best_score = acc
                    best_params = {
                        "max_depth": max_depth,
                        "min_samples_split": min_samples_split,
                        "min_samples_leaf": min_samples_leaf,
                        "criterion": criterion
                    }
                    best_model = dt

end_search = time.perf_counter()
cpu_after_search = process.cpu_times()

search_time = end_search - start_search
search_cpu_user = cpu_after_search.user - cpu_before_search.user
search_cpu_sys  = cpu_after_search.system - cpu_before_search.system

print(f"Recherche : {n_combinations} combinaisons testées")
print(f"Meilleure accuracy : {best_score:.4f}")
print(f"Meilleurs params   : {best_params}")
print(f"Temps total recherche : {search_time:.2f} s "
      f"(CPU user {search_cpu_user:.2f}s | sys {search_cpu_sys:.2f}s)")
print(f"Temps moyen / combinaison : {search_time / n_combinations:.3f} s\n")

# --- Évaluation détaillée du meilleur modèle ---
cpu_before_fit = process.cpu_times()
start_fit = time.perf_counter()
best_model.fit(X_train_svd, y_train)
end_fit = time.perf_counter()
cpu_after_fit = process.cpu_times()

fit_time = end_fit - start_fit
fit_cpu_user = cpu_after_fit.user - cpu_before_fit.user
fit_cpu_sys  = cpu_after_fit.system - cpu_before_fit.system

cpu_before_pred = process.cpu_times()
start_pred = time.perf_counter()
y_pred_dt = best_model.predict(X_test_svd)
end_pred = time.perf_counter()
cpu_after_pred = process.cpu_times()

pred_time = end_pred - start_pred
pred_cpu_user = cpu_after_pred.user - cpu_before_pred.user
pred_cpu_sys  = cpu_after_pred.system - cpu_before_pred.system

print("Classification (best model) : ")
print(f"Accuracy : {accuracy_score(y_test, y_pred_dt):.4f}")
print(classification_report(y_test, y_pred_dt))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred_dt))

mse  = mean_squared_error(y_test, y_pred_dt)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred_dt)

print("\nErrors : ")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

print("\nPerformance (best model) :")
print(f"Entraînement : {fit_time:.4f} s "
      f"(CPU user {fit_cpu_user:.4f}s | sys {fit_cpu_sys:.4f}s)")
print(f"Inférence    : {pred_time:.4f} s "
      f"(CPU user {pred_cpu_user:.4f}s | sys {pred_cpu_sys:.4f}s)")
print(f"Total : {fit_time + pred_time:.4f} s ")
print(f"\nArbre : profondeur={best_model.get_depth()}, "
      f"nb feuilles={best_model.get_n_leaves()}")

Recherche : 160 combinaisons testées
Meilleure accuracy : 0.6353
Meilleurs params   : {'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 8, 'criterion': 'entropy'}
Temps total recherche : 85.67 s (CPU user 83.48s | sys 1.25s)
Temps moyen / combinaison : 0.535 s

Classification (best model) : 
Accuracy : 0.6353
              precision    recall  f1-score   support

           0       0.35      0.32      0.34       121
           1       0.71      0.84      0.77       575
           2       0.53      0.34      0.41       272

    accuracy                           0.64       968
   macro avg       0.53      0.50      0.51       968
weighted avg       0.61      0.64      0.61       968

Confusion matrix :
[[ 39  54  28]
 [ 36 484  55]
 [ 35 145  92]]

Errors : 
MSE  : 0.5599
RMSE : 0.7483
MAE  : 0.4298

Performance (best model) :
Entraînement : 0.6126 s (CPU user 0.5938s | sys 0.0156s)
Inférence    : 0.0004 s (CPU user 0.0000s | sys 0.0000s)
Total : 0.6130 s 

Arbre : profondeu

# **Part 3 : Classifier 2.0**

## **3.1 - Import Data Set**

Utilisation de df_train_original et df_test_original

## **3.2 - Data Cleaning**

### Text Cleaning

In [24]:
nltk.download('stopwords')

# Stopwords anglais
stop_words = set(stopwords.words('english'))

negations = {"not", "no", "never"}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    tokens = text.split()
    
    # garder les négations même si elles sont dans stopwords
    tokens = [word for word in tokens if (word not in stop_words or word in negations)]
    
    return " ".join(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jerem\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Lemmatize Text

In [25]:
nlp = spacy.load("en_core_web_sm")

def lemmatize_text(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_punct]
    return " ".join(tokens)

In [26]:
df_train["sentence_clean"] = df_train_original["sentence"].apply(clean_text).apply(lemmatize_text)
df_test["sentence_clean"] = df_test_original["sentence"].apply(clean_text).apply(lemmatize_text)

In [27]:
negations = {"not", "no", "never", "none", "didn't", "nobody", "nothing", "nowhere", "neither", "nor"}

count = 0

for text in df_train["sentence_clean"]:
    words = text.split()
    count += sum(1 for w in words if w in negations)

print("Number of negations :", count)

Number of negations : 133


### Vectorizing Text

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(df_train["sentence_clean"])
X_test_tfidf = tfidf_vectorizer.transform(df_test["sentence_clean"])


Top 5 most used words

In [29]:
import numpy as np

# moyenne des scores TF-IDF pour chaque mot (colonne)
mean_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).ravel()

# récupérer les mots
feature_names = tfidf_vectorizer.get_feature_names_out()

# trier du plus important au moins important
top_indices = mean_tfidf.argsort()[::-1]

# afficher les 5 mots les plus importants
for i in top_indices[:5]:
    print(feature_names[i], mean_tfidf[i])

eur 0.046826891094503234
mn 0.030239243395692957
company 0.02596816878304087
profit 0.018818566404265463
sale 0.018778526298372976


### Dimension Reduction

In [30]:
from sklearn.decomposition import TruncatedSVD

# Réduction de dimension
svd = TruncatedSVD(n_components=170, random_state=42)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)

print(X_train_svd.shape)
print(X_test_svd.shape)

(3872, 170)
(968, 170)


## **3.3 - Logistic Regression 2.0**

In [31]:
y_train = df_train["label"]
y_test = df_test["label"]

process = psutil.Process(os.getpid())
cpu_before_fit = process.cpu_times()

clf = LogisticRegression(max_iter=1000, random_state=42)
start_fit = time.perf_counter()
clf.fit(X_train_svd, y_train)
end_fit = time.perf_counter()

cpu_after_fit = process.cpu_times()

fit_time = end_fit - start_fit
fit_cpu_user = cpu_after_fit.user - cpu_before_fit.user
fit_cpu_sys  = cpu_after_fit.system - cpu_before_fit.system

cpu_before_pred = process.cpu_times()

start_pred = time.perf_counter()
y_pred = clf.predict(X_test_svd)
end_pred = time.perf_counter()

cpu_after_pred = process.cpu_times()

pred_time = end_pred - start_pred
pred_cpu_user = cpu_after_pred.user - cpu_before_pred.user
pred_cpu_sys  = cpu_after_pred.system - cpu_before_pred.system

print("Classification : ")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred))

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)

print("\nErrors : ")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

print("\nPerformance :")
print(f"Entraînement : {fit_time:.4f} s "
      f"(CPU user {fit_cpu_user:.4f}s | sys {fit_cpu_sys:.4f}s)")
print(f"Inférence    : {pred_time:.4f} s "
      f"(CPU user {pred_cpu_user:.4f}s | sys {pred_cpu_sys:.4f}s)")
print(f"Total : {fit_time + pred_time:.4f} s ")

Classification : 
Accuracy : 0.6963
              precision    recall  f1-score   support

           0       0.77      0.30      0.43       121
           1       0.70      0.94      0.80       575
           2       0.66      0.35      0.46       272

    accuracy                           0.70       968
   macro avg       0.71      0.53      0.56       968
weighted avg       0.70      0.70      0.66       968

Confusion matrix :
[[ 36  66  19]
 [  3 542  30]
 [  8 168  96]]

Errors : 
MSE  : 0.3874
RMSE : 0.6224
MAE  : 0.3316

Performance :
Entraînement : 0.0358 s (CPU user 0.0156s | sys 0.1406s)
Inférence    : 0.0004 s (CPU user 0.0000s | sys 0.0000s)
Total : 0.0361 s 


## **3.4 - Gaussian Classifier 2.0**

In [32]:
process = psutil.Process(os.getpid())
cpu_before_fit = process.cpu_times()

gnb = GaussianNB()
start_fit = time.perf_counter()
gnb.fit(X_train_svd, y_train)
end_fit = time.perf_counter()

cpu_after_fit = process.cpu_times()

fit_time = end_fit - start_fit
fit_cpu_user = cpu_after_fit.user - cpu_before_fit.user
fit_cpu_sys  = cpu_after_fit.system - cpu_before_fit.system

cpu_before_pred = process.cpu_times()

start_pred = time.perf_counter()
y_pred_gnb = gnb.predict(X_test_svd)
end_pred = time.perf_counter()

cpu_after_pred = process.cpu_times()

pred_time = end_pred - start_pred
pred_cpu_user = cpu_after_pred.user - cpu_before_pred.user
pred_cpu_sys  = cpu_after_pred.system - cpu_before_pred.system

print("Classification : ")
print(f"Accuracy : {accuracy_score(y_test, y_pred_gnb):.4f}")
print(classification_report(y_test, y_pred_gnb))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred_gnb))

mse  = mean_squared_error(y_test, y_pred_gnb)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred_gnb)

print("\nErrors : ")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

print("\nPerformance :")
print(f"Entraînement : {fit_time:.4f} s "
      f"(CPU user {fit_cpu_user:.4f}s | sys {fit_cpu_sys:.4f}s)")
print(f"Inférence    : {pred_time:.4f} s "
      f"(CPU user {pred_cpu_user:.4f}s | sys {pred_cpu_sys:.4f}s)")
print(f"Total : {fit_time + pred_time:.4f} s ")

Classification : 
Accuracy : 0.6116
              precision    recall  f1-score   support

           0       0.33      0.38      0.35       121
           1       0.68      0.82      0.74       575
           2       0.55      0.28      0.37       272

    accuracy                           0.61       968
   macro avg       0.52      0.49      0.49       968
weighted avg       0.60      0.61      0.59       968

Confusion matrix :
[[ 46  61  14]
 [ 58 470  47]
 [ 36 160  76]]

Errors : 
MSE  : 0.5434
RMSE : 0.7371
MAE  : 0.4401

Performance :
Entraînement : 0.0047 s (CPU user 0.0312s | sys 0.0000s)
Inférence    : 0.0011 s (CPU user 0.0000s | sys 0.0000s)
Total : 0.0058 s 


## **3.5 - Bernoulli Classifier 2.0**

In [33]:
# Transformer TF-IDF en binaire
X_train_bin = (X_train_tfidf > 0).astype(int)
X_test_bin = (X_test_tfidf > 0).astype(int)

process = psutil.Process(os.getpid())
cpu_before_fit = process.cpu_times()

bnb = BernoulliNB()
start_fit = time.perf_counter()
bnb.fit(X_train_bin, y_train)
end_fit = time.perf_counter()

cpu_after_fit = process.cpu_times()

fit_time = end_fit - start_fit
fit_cpu_user = cpu_after_fit.user - cpu_before_fit.user
fit_cpu_sys  = cpu_after_fit.system - cpu_before_fit.system

cpu_before_pred = process.cpu_times()

start_pred = time.perf_counter()
y_pred_bnb = bnb.predict(X_test_bin)
end_pred = time.perf_counter()

cpu_after_pred = process.cpu_times()

pred_time = end_pred - start_pred
pred_cpu_user = cpu_after_pred.user - cpu_before_pred.user
pred_cpu_sys  = cpu_after_pred.system - cpu_before_pred.system

print("Classification : ")
print(f"Accuracy : {accuracy_score(y_test, y_pred_bnb):.4f}")
print(classification_report(y_test, y_pred_bnb))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred_bnb))

mse  = mean_squared_error(y_test, y_pred_bnb)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred_bnb)

print("\nErrors : ")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

print("\nPerformance :")
print(f"Entraînement : {fit_time:.4f} s "
      f"(CPU user {fit_cpu_user:.4f}s | sys {fit_cpu_sys:.4f}s)")
print(f"Inférence    : {pred_time:.4f} s "
      f"(CPU user {pred_cpu_user:.4f}s | sys {pred_cpu_sys:.4f}s)")
print(f"Total : {fit_time + pred_time:.4f} s ")

Classification : 
Accuracy : 0.6880
              precision    recall  f1-score   support

           0       0.89      0.07      0.12       121
           1       0.71      0.94      0.81       575
           2       0.58      0.43      0.50       272

    accuracy                           0.69       968
   macro avg       0.73      0.48      0.48       968
weighted avg       0.70      0.69      0.64       968

Confusion matrix :
[[  8  64  49]
 [  0 541  34]
 [  1 154 117]]

Errors : 
MSE  : 0.4669
RMSE : 0.6833
MAE  : 0.3636

Performance :
Entraînement : 0.0015 s (CPU user 0.0000s | sys 0.0000s)
Inférence    : 0.0005 s (CPU user 0.0000s | sys 0.0000s)
Total : 0.0019 s 


## **3.6 - Decision Tree 2.0**

In [34]:
best_score = 0
best_params = None
best_model = None

process = psutil.Process(os.getpid())
cpu_before_search = process.cpu_times()
start_search = time.perf_counter()

n_combinations = 0

for max_depth in [5, 10, 15, 20, None]:
    for min_samples_split in [2, 5, 10, 20]:
        for min_samples_leaf in [1, 2, 4, 8]:
            for criterion in ["gini", "entropy"]:
                n_combinations += 1

                dt = DecisionTreeClassifier(
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    min_samples_leaf=min_samples_leaf,
                    criterion=criterion,
                    random_state=42
                )

                dt.fit(X_train_svd, y_train)
                y_pred = dt.predict(X_test_svd)
                acc = accuracy_score(y_test, y_pred)

                if acc > best_score:
                    best_score = acc
                    best_params = {
                        "max_depth": max_depth,
                        "min_samples_split": min_samples_split,
                        "min_samples_leaf": min_samples_leaf,
                        "criterion": criterion
                    }
                    best_model = dt

end_search = time.perf_counter()
cpu_after_search = process.cpu_times()

search_time = end_search - start_search
search_cpu_user = cpu_after_search.user - cpu_before_search.user
search_cpu_sys  = cpu_after_search.system - cpu_before_search.system

print(f"Recherche : {n_combinations} combinaisons testées")
print(f"Meilleure accuracy : {best_score:.4f}")
print(f"Meilleurs params   : {best_params}")
print(f"Temps total recherche : {search_time:.2f} s "
      f"(CPU user {search_cpu_user:.2f}s | sys {search_cpu_sys:.2f}s)")
print(f"Temps moyen / combinaison : {search_time / n_combinations:.3f} s\n")

# --- Évaluation détaillée du meilleur modèle ---
cpu_before_fit = process.cpu_times()
start_fit = time.perf_counter()
best_model.fit(X_train_svd, y_train)
end_fit = time.perf_counter()
cpu_after_fit = process.cpu_times()

fit_time = end_fit - start_fit
fit_cpu_user = cpu_after_fit.user - cpu_before_fit.user
fit_cpu_sys  = cpu_after_fit.system - cpu_before_fit.system

cpu_before_pred = process.cpu_times()
start_pred = time.perf_counter()
y_pred_dt = best_model.predict(X_test_svd)
end_pred = time.perf_counter()
cpu_after_pred = process.cpu_times()

pred_time = end_pred - start_pred
pred_cpu_user = cpu_after_pred.user - cpu_before_pred.user
pred_cpu_sys  = cpu_after_pred.system - cpu_before_pred.system

print("Classification (best model) : ")
print(f"Accuracy : {accuracy_score(y_test, y_pred_dt):.4f}")
print(classification_report(y_test, y_pred_dt))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred_dt))

mse  = mean_squared_error(y_test, y_pred_dt)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred_dt)

print("\nErrors : ")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

print("\nPerformance (best model) :")
print(f"Entraînement : {fit_time:.4f} s "
      f"(CPU user {fit_cpu_user:.4f}s | sys {fit_cpu_sys:.4f}s)")
print(f"Inférence    : {pred_time:.4f} s "
      f"(CPU user {pred_cpu_user:.4f}s | sys {pred_cpu_sys:.4f}s)")
print(f"Total : {fit_time + pred_time:.4f} s ")
print(f"\nArbre : profondeur={best_model.get_depth()}, "
      f"nb feuilles={best_model.get_n_leaves()}")

Recherche : 160 combinaisons testées
Meilleure accuracy : 0.6457
Meilleurs params   : {'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 1, 'criterion': 'entropy'}
Temps total recherche : 83.65 s (CPU user 82.36s | sys 0.56s)
Temps moyen / combinaison : 0.523 s

Classification (best model) : 
Accuracy : 0.6457
              precision    recall  f1-score   support

           0       0.56      0.15      0.24       121
           1       0.68      0.90      0.77       575
           2       0.51      0.34      0.41       272

    accuracy                           0.65       968
   macro avg       0.59      0.46      0.47       968
weighted avg       0.62      0.65      0.60       968

Confusion matrix :
[[ 18  70  33]
 [  6 515  54]
 [  8 172  92]]

Errors : 
MSE  : 0.4814
RMSE : 0.6938
MAE  : 0.3967

Performance (best model) :
Entraînement : 0.3850 s (CPU user 0.3594s | sys 0.0000s)
Inférence    : 0.0004 s (CPU user 0.0000s | sys 0.0000s)
Total : 0.3854 s 

Arbre : profondeur

## **3.7 - Random Forest 2.0**

In [35]:
from sklearn.ensemble import RandomForestClassifier

best_score = 0
best_params = None
best_model = None

process = psutil.Process(os.getpid())
cpu_before_search = process.cpu_times()
start_search = time.perf_counter()

n_combinations = 0

for n_estimators in [50, 100, 200]:
    for max_depth in [5, 10, 20, None]:
        for max_features in ["sqrt", "log2"]:
            for min_samples_split in [2, 5, 10]:
                for min_samples_leaf in [1, 2, 4]:
                    n_combinations += 1

                    rf = RandomForestClassifier(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        max_features=max_features,
                        min_samples_split=min_samples_split,
                        min_samples_leaf=min_samples_leaf,
                        random_state=42,
                        n_jobs=-1
                    )

                    rf.fit(X_train_svd, y_train)
                    y_pred = rf.predict(X_test_svd)
                    acc = accuracy_score(y_test, y_pred)

                    if acc > best_score:
                        best_score = acc
                        best_params = {
                            "n_estimators": n_estimators,
                            "max_depth": max_depth,
                            "max_features": max_features,
                            "min_samples_split": min_samples_split,
                            "min_samples_leaf": min_samples_leaf
                        }
                        best_model = rf

end_search = time.perf_counter()
cpu_after_search = process.cpu_times()

search_time = end_search - start_search
search_cpu_user = cpu_after_search.user - cpu_before_search.user
search_cpu_sys  = cpu_after_search.system - cpu_before_search.system

print(f"Recherche : {n_combinations} combinaisons testées")
print(f"Meilleure accuracy : {best_score:.4f}")
print(f"Meilleurs params   : {best_params}")
print(f"Temps total recherche : {search_time:.2f} s "
      f"(CPU user {search_cpu_user:.2f}s | sys {search_cpu_sys:.2f}s)")
print(f"Temps moyen / combinaison : {search_time / n_combinations:.3f} s\n")

# --- Évaluation détaillée du meilleur modèle ---
cpu_before_fit = process.cpu_times()
start_fit = time.perf_counter()
best_model.fit(X_train_svd, y_train)
end_fit = time.perf_counter()
cpu_after_fit = process.cpu_times()

fit_time = end_fit - start_fit
fit_cpu_user = cpu_after_fit.user - cpu_before_fit.user
fit_cpu_sys  = cpu_after_fit.system - cpu_before_fit.system

cpu_before_pred = process.cpu_times()
start_pred = time.perf_counter()
y_pred_rf = best_model.predict(X_test_svd)
end_pred = time.perf_counter()
cpu_after_pred = process.cpu_times()

pred_time = end_pred - start_pred
pred_cpu_user = cpu_after_pred.user - cpu_before_pred.user
pred_cpu_sys  = cpu_after_pred.system - cpu_before_pred.system

print("Classification (best model) : ")
print(f"Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}")
print(classification_report(y_test, y_pred_rf))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred_rf))

mse  = mean_squared_error(y_test, y_pred_rf)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred_rf)

print("\nErrors : ")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

print("\nPerformance (best model) :")
print(f"Entraînement : {fit_time:.4f} s "
      f"(CPU user {fit_cpu_user:.4f}s | sys {fit_cpu_sys:.4f}s)")
print(f"Inférence    : {pred_time:.4f} s "
      f"(CPU user {pred_cpu_user:.4f}s | sys {pred_cpu_sys:.4f}s)")
print(f"Total : {fit_time + pred_time:.4f} s ")
print(f"\nForêt : {best_model.n_estimators} arbres, "
      f"profondeur moyenne ≈ {np.mean([t.get_depth() for t in best_model.estimators_]):.1f}")

Recherche : 216 combinaisons testées
Meilleure accuracy : 0.6952
Meilleurs params   : {'n_estimators': 100, 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 2, 'min_samples_leaf': 1}
Temps total recherche : 60.39 s (CPU user 675.30s | sys 79.11s)
Temps moyen / combinaison : 0.280 s

Classification (best model) : 
Accuracy : 0.6952
              precision    recall  f1-score   support

           0       0.72      0.26      0.38       121
           1       0.69      0.96      0.80       575
           2       0.72      0.32      0.45       272

    accuracy                           0.70       968
   macro avg       0.71      0.51      0.54       968
weighted avg       0.70      0.70      0.65       968

Confusion matrix :
[[ 31  69  21]
 [  7 554  14]
 [  5 179  88]]

Errors : 
MSE  : 0.3853
RMSE : 0.6208
MAE  : 0.3316

Performance (best model) :
Entraînement : 0.3614 s (CPU user 4.8906s | sys 0.3594s)
Inférence    : 0.0165 s (CPU user 0.0312s | sys 0.0000s)
Total : 0.377

# **Part 4 : Classifier 3.0**

## **4.1 - Import Data Set**

Utilisation de df_train_original et df_test_original

## **4.2 - Data Cleaning**

### Text Cleaning

In [36]:
def clean_text_bert(text):
    text = str(text).lower().strip()
    return text

In [37]:
df_train["sentence_clean"] = df_train_original["sentence"].apply(clean_text_bert)
df_test["sentence_clean"] = df_test_original["sentence"].apply(clean_text_bert)

## **4.3 - Sentence BERT Classifier**

In [38]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

y_train = df_train["label"]
y_test = df_test["label"]

process = psutil.Process(os.getpid())

# --- 1) Chargement du modèle d'embeddings ---
start_load = time.perf_counter()
embedder = SentenceTransformer("all-MiniLM-L6-v2")
end_load = time.perf_counter()
load_time = end_load - start_load

# --- 2) Encodage train ---
cpu_before_enc_train = process.cpu_times()
start_enc_train = time.perf_counter()
X_train_emb = embedder.encode(df_train["sentence"].tolist(), show_progress_bar=True)
end_enc_train = time.perf_counter()
cpu_after_enc_train = process.cpu_times()

enc_train_time = end_enc_train - start_enc_train
enc_train_cpu_user = cpu_after_enc_train.user - cpu_before_enc_train.user
enc_train_cpu_sys  = cpu_after_enc_train.system - cpu_before_enc_train.system

# --- 3) Encodage test ---
cpu_before_enc_test = process.cpu_times()
start_enc_test = time.perf_counter()
X_test_emb = embedder.encode(df_test["sentence"].tolist(), show_progress_bar=True)
end_enc_test = time.perf_counter()
cpu_after_enc_test = process.cpu_times()

enc_test_time = end_enc_test - start_enc_test
enc_test_cpu_user = cpu_after_enc_test.user - cpu_before_enc_test.user
enc_test_cpu_sys  = cpu_after_enc_test.system - cpu_before_enc_test.system

# --- 4) Entraînement classifieur ---
cpu_before_fit = process.cpu_times()
start_fit = time.perf_counter()
clf = LogisticRegression(max_iter=2000, random_state=42)
clf.fit(X_train_emb, y_train)
end_fit = time.perf_counter()
cpu_after_fit = process.cpu_times()

fit_time = end_fit - start_fit
fit_cpu_user = cpu_after_fit.user - cpu_before_fit.user
fit_cpu_sys  = cpu_after_fit.system - cpu_before_fit.system

# --- 5) Prédiction ---
cpu_before_pred = process.cpu_times()
start_pred = time.perf_counter()
y_pred = clf.predict(X_test_emb)
end_pred = time.perf_counter()
cpu_after_pred = process.cpu_times()

pred_time = end_pred - start_pred
pred_cpu_user = cpu_after_pred.user - cpu_before_pred.user
pred_cpu_sys  = cpu_after_pred.system - cpu_before_pred.system

# --- Résultats ---
print("Classification : ")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred))

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)

print("\nErrors : ")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

print("\nPerformance :")
print(f"Chargement modèle      : {load_time:.4f} s")
print(f"Encodage train ({len(df_train)} ex.) : {enc_train_time:.4f} s "
      f"(CPU user {enc_train_cpu_user:.4f}s | sys {enc_train_cpu_sys:.4f}s)")
print(f"Encodage test  ({len(df_test)} ex.) : {enc_test_time:.4f} s "
      f"(CPU user {enc_test_cpu_user:.4f}s | sys {enc_test_cpu_sys:.4f}s)")
print(f"Entraînement classif.  : {fit_time:.4f} s "
      f"(CPU user {fit_cpu_user:.4f}s | sys {fit_cpu_sys:.4f}s)")
print(f"Inférence classif.     : {pred_time:.4f} s "
      f"(CPU user {pred_cpu_user:.4f}s | sys {pred_cpu_sys:.4f}s)")

print(f"Total : {load_time + enc_train_time + enc_test_time + fit_time + pred_time:.4f} s ")

print(f"\nDimension embeddings : {X_train_emb.shape[1]}")

Batches: 100%|██████████| 31/31 [00:01<00:00, 17.09it/s]

Classification : 
Accuracy : 0.7748
              precision    recall  f1-score   support

           0       0.78      0.66      0.72       121
           1       0.79      0.90      0.84       575
           2       0.73      0.56      0.63       272

    accuracy                           0.77       968
   macro avg       0.77      0.71      0.73       968
weighted avg       0.77      0.77      0.77       968

Confusion matrix :
[[ 80  29  12]
 [ 12 517  46]
 [ 10 109 153]]

Errors : 
MSE  : 0.2934
RMSE : 0.5417
MAE  : 0.2479

Performance :
Chargement modèle      : 3.3449 s
Encodage train (3872 ex.) : 7.5497 s (CPU user 96.5156s | sys 6.2031s)
Encodage test  (968 ex.) : 1.8210 s (CPU user 23.6406s | sys 1.3906s)
Entraînement classif.  : 0.0916 s (CPU user 0.9219s | sys 0.2344s)
Inférence classif.     : 0.0013 s (CPU user 0.2031s | sys 0.0156s)
Total : 12.8085 s 

Dimension embeddings : 384


# **Part 5 : Classifier 4.0**

## **5.2 - Data Cleaning**

### Text Cleaning

In [39]:
def clean_text_bert(text):
    text = str(text).lower().strip()
    return text

In [40]:
df_train["sentence_clean"] = df_train_original["sentence"].apply(clean_text_bert)
df_test["sentence_clean"] = df_test_original["sentence"].apply(clean_text_bert)

## **5.3 - Fine-Tuning BERT Transformers Classifier**

In [41]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate
import torch

process = psutil.Process(os.getpid())

# --- 1) Préparer les données ---
train_ds = Dataset.from_pandas(df_train[["sentence", "label"]])
test_ds = Dataset.from_pandas(df_test[["sentence", "label"]])

# --- 2) Tokenizer + modèle ---
model_name = "distilbert-base-uncased"
start_load = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)
load_time = time.perf_counter() - start_load

# --- 3) Tokenisation ---
def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True)

start_tok = time.perf_counter()
train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)
tok_time = time.perf_counter() - start_tok

train_ds = train_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# --- 4) Métriques ---
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

# --- 5) Entraînement ---
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Mémoire avant fit
mem_before_fit = process.memory_info().rss / 1024**2
cpu_before_fit = process.cpu_times()
start_fit = time.perf_counter()

train_result = trainer.train()

end_fit = time.perf_counter()
cpu_after_fit = process.cpu_times()
mem_after_fit = process.memory_info().rss / 1024**2

fit_time = end_fit - start_fit
fit_cpu_user = cpu_after_fit.user - cpu_before_fit.user
fit_cpu_sys  = cpu_after_fit.system - cpu_before_fit.system

# --- 6) Prédiction ---
cpu_before_pred = process.cpu_times()
start_pred = time.perf_counter()
pred_output = trainer.predict(test_ds)
end_pred = time.perf_counter()
cpu_after_pred = process.cpu_times()

pred_time = end_pred - start_pred
pred_cpu_user = cpu_after_pred.user - cpu_before_pred.user
pred_cpu_sys  = cpu_after_pred.system - cpu_before_pred.system

y_pred_bert = np.argmax(pred_output.predictions, axis=1)
y_test_bert = pred_output.label_ids

# --- 7) Métriques ---
print("Classification : ")
print(f"Accuracy : {accuracy_score(y_test_bert, y_pred_bert):.4f}")
print(classification_report(y_test_bert, y_pred_bert))
print("Confusion matrix :")
print(confusion_matrix(y_test_bert, y_pred_bert))

mse  = mean_squared_error(y_test_bert, y_pred_bert)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_bert, y_pred_bert)

print("\nErrors : ")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

print("\nPerformance :")
print(f"Chargement modèle    : {load_time:.4f} s")
print(f"Tokenisation         : {tok_time:.4f} s")
print(f"Entraînement (fine-tuning) : {fit_time:.2f} s "
      f"(CPU user {fit_cpu_user:.2f}s | sys {fit_cpu_sys:.2f}s)")
print(f"Inférence sur test   : {pred_time:.4f} s "
      f"(CPU user {pred_cpu_user:.4f}s | sys {pred_cpu_sys:.4f}s)")
print(f"Temps moyen / échantillon (inférence) : {pred_time / len(test_ds) * 1000:.4f} ms")

print("\nMémoire :")
print(f"RAM avant fit : {mem_before_fit:.1f} Mo")
print(f"RAM après fit : {mem_after_fit:.1f} Mo  (Δ {mem_after_fit - mem_before_fit:+.1f} Mo)")

# GPU si dispo
if torch.cuda.is_available():
    print(f"GPU mémoire allouée : {torch.cuda.memory_allocated() / 1024**2:.1f} Mo")
    print(f"GPU mémoire pic     : {torch.cuda.max_memory_allocated() / 1024**2:.1f} Mo")
elif torch.backends.mps.is_available():
    print(f"Device : MPS (Apple Silicon)")
else:
    print(f"Device : CPU")

# Métriques Trainer (loss, samples/sec, etc.)
print("\nMétriques d'entraînement (Trainer) :")
for k, v in train_result.metrics.items():
    print(f"  {k}: {v}")

print(f"\nModèle : {model_name}")
print(f"Nb paramètres : {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10645.44it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 968/968 [00:00<00:00, 42968.42 examples/s]
C:\Users\jerem\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfr

Epoch,Training Loss,Validation Loss,Accuracy
1,0.431460,0.401303,0.832645


C:\Users\jerem\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Classification : 
Accuracy : 0.8326
              precision    recall  f1-score   support

           0       0.74      0.84      0.79       121
           1       0.89      0.86      0.87       575
           2       0.77      0.77      0.77       272

    accuracy                           0.83       968
   macro avg       0.80      0.82      0.81       968
weighted avg       0.84      0.83      0.83       968

Confusion matrix :
[[102  11   8]
 [ 24 495  56]
 [ 11  52 209]]

Errors : 
MSE  : 0.2262
RMSE : 0.4756
MAE  : 0.1870

Performance :
Chargement modèle    : 1.0244 s
Tokenisation         : 0.2224 s
Entraînement (fine-tuning) : 163.95 s (CPU user 2023.59s | sys 186.91s)
Inférence sur test   : 8.7087 s (CPU user 111.7031s | sys 6.0469s)
Temps moyen / échantillon (inférence) : 8.9966 ms

Mémoire :
RAM avant fit : 1052.5 Mo
RAM après fit : 2069.2 Mo  (Δ +1016.7 Mo)
Device : CPU

Métriques d'entraînement (Trainer) :
  train_runtime: 163.8077
  train_samples_per_second: 23.637
  trai